In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from functools import reduce
from pathlib import Path
from dotenv import load_dotenv
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from datetime import datetime
from itertools import combinations
from pandas.api.types import is_integer_dtype
from collections import Counter, defaultdict

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
print("hello")

In [ ]:
# -------------------------------------------------------------------------
# Bulk CSV Loader (flat only, does not open subfolders of data) with Smart Encoding Handling
#
# What this script does:
# 1. Sets input (RAW_DIR) and output (OUT_DIR) directories.
# 2. Finds and loads all CSV files in RAW_DIR *only* (NO subfolders).
# 3. Handles tricky encodings (UTF-8 with BOM, fallback to Latin-1).
# 4. Keeps loading even if some files fail, and reports errors.
# 5. Stores loaded CSVs in a dictionary (tables) keyed by file name (no .csv).
# 6. Prints a summary: number of CSVs loaded and their names.
# -------------------------------------------------------------------------

# Define input and output directories
current_user = os.getlogin()
if current_user == "bouba.ismalia":
    RAW_DIR = Path(rf"C:\Users\{current_user}\Stichting Hogeschool Utrecht\FCA-DA-P - data")
else:
    RAW_DIR = Path(rf"C:\Users\{current_user}\Stichting Hogeschool Utrecht\FCA-DA-P - Analytics\Student drop-out\data")

OUT_DIR = Path(rf"{RAW_DIR}\processed")

# --- Helpers ---
def read_csv_smart(path: Path, **kwargs) -> pd.DataFrame:
    """Read CSV with sensible defaults, trying UTF-8 first and Latin-1 as fallback."""
    try:
        return pd.read_csv(path, encoding="utf-8-sig", **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", **kwargs)

def load_csvs_flat(base: Path, **kwargs) -> dict[str, pd.DataFrame]:
    """
    Load CSVs located directly in `base` (no subfolders) into a dict keyed by filename (without .csv).
    Prints how many tables were loaded and how many errors occurred.
    """
    tables, errors = {}, []
    for p in sorted(base.glob("*.csv")):  # <-- flat search only
        key = p.stem  # filename without extension
        try:
            tables[key] = read_csv_smart(p, **kwargs)
        except Exception as e:
            errors.append((str(p), str(e)))
    print(f"Loaded {len(tables)} tables; errors: {len(errors)}")
    if errors:
        for name, err in errors[:10]:
            print(" -", name, "->", err)
    return tables

# --- Load CSV files in RAW_DIR (flat only) ---
tables = load_csvs_flat(
    RAW_DIR,
    sep=None,        # auto-detect delimiter
    engine="python", # required for sep=None sniffing
)

# Print summary in the requested format
print(f"\nNumber of csv files found: {len(tables)}")
print("List of csv files:")
for i, name in enumerate(tables.keys(), start=1):
    print(f"{i}. {name}")


In [ ]:
# show summary of data files
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_seq_items", None)   # don't truncate lists

summary = (
    pd.DataFrame(
        [{
            "key": k,
            "rows": len(df),
            "cols": df.shape[1],
            "columns": list(map(str, df.columns))  # full list, no manual truncation
        } for k, df in tables.items()]
    ).sort_values("key")
)

summary


In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Create globally accessible pandas DataFrames with clean, consistent
# names from a `tables` source dict. For each (clean_name -> source_key) pair,
# this script fetches the DataFrame from `tables`, assigns it into `globals()`
# under `clean_name`, and logs what was created. If a source key is missing,
# it logs a warning.
# -----------------------------------------------------------------------------

# Step 1: Map clean DataFrame names -> source keys in `tables`
custom_names = {
    "sdo1_distance": "sdo1-helperdata-Euclidean_distance_postcode_to_HU",
    "sdo1_characteristics": "sdo1-student_characteristics",
    "sdo1_previous": "sdo1-student_previous_education",
    "sdo2_skc": "sdo2-skc",
    "sdo2_orientation": "sdo2-student_orientation",
    "sdo4_dsa": "sdo4-DSA_degree_collegeyear_2018-2023",
    "sdo5_degree": "sdo5-student_degree_drop-out_postalcode",
    "sdo6_results": "sdo6-course_results",
    "sdo7_questionaire": "sdo78-student_wellbeing",
}

# Step 2: Create DataFrames with clean names (keep everything)
for clean_name, source_key in custom_names.items():
    df = tables.get(source_key)
    if df is None:
        print(f"[warning] '{source_key}' not found in tables.")
        continue

    globals()[clean_name] = df
    print(f"Created DataFrame: {clean_name} from '{source_key}' with {len(df)} rows")


------------------------- For each dataframe, drop rows that are entirely empty across all columns ----------------------------------

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: For each loaded DataFrame, convert empty/whitespace-only strings to NaN
# (object/string columns only) and DROP rows that are entirely empty (all-NaN).
# Prints a before/after summary per dataset. Safe to run multiple times.
# -----------------------------------------------------------------------------

df_names = [
    "sdo1_distance",
    "sdo1_characteristics",
    "sdo1_previous",
    "sdo2_skc",
    "sdo2_orientation",
    "sdo4_dsa",
    "sdo5_degree",
    "sdo6_results",
    "sdo7_questionaire",
]

def _drop_fully_empty_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    # Treat pure-whitespace as empty only for textual columns
    obj_cols = df.select_dtypes(include=["object", "string"]).columns
    if len(obj_cols):
        df = df.copy()
        df[obj_cols] = df[obj_cols].replace(r"^\s*$", np.nan, regex=True)

    empty_mask = df.isna().all(axis=1)
    n_empty = int(empty_mask.sum())
    return df.loc[~empty_mask].copy() if n_empty else df, n_empty

for name in df_names:
    df = globals().get(name)
    if not isinstance(df, pd.DataFrame):
        print(f"[warning] '{name}' not found or not a DataFrame; skipping.")
        continue

    before = len(df)
    cleaned, dropped = _drop_fully_empty_rows(df)
    globals()[name] = cleaned
    after = len(cleaned)

    print(f"{name}: dropped {dropped} fully-empty rows (rows {before} → {after})")

# Assert no fully empty rows remain in any dataset
for name in df_names:
    df = globals().get(name)
    if isinstance(df, pd.DataFrame):
        assert not df.isna().all(axis=1).any(), f"{name} still has fully empty rows."


Anticipation: 

Fully empty rows usually come from source files (especially Excel) where the sheet’s UsedRange includes formatted-but-empty rows or trailing delimiters, 
so the importer reads them as blank records. They can also be created downstream by outer joins/concats with mismatched keys or 
by converting whitespace/empty strings to NaN across all columns, leaving rows with nothing but missing values.

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Drop any "unnamed" columns (e.g., "Unnamed: 0", empty-string headers)
# from all loaded DataFrames. Safe to run multiple times; prints what was removed.
# -----------------------------------------------------------------------------

df_names = [
    "sdo1_distance",
    "sdo1_characteristics",
    "sdo1_previous",
    "sdo2_skc",
    "sdo2_orientation",
    "sdo4_dsa",
    "sdo5_degree",
    "sdo6_results",
    "sdo7_questionaire",
]

def _unnamed_cols(df: pd.DataFrame):
    cols = []
    for c in df.columns:
        s = str(c)
        if s.strip() == "" or s.lower().startswith("unnamed"):
            cols.append(c)
    return cols

for name in df_names:
    df = globals().get(name)
    if not isinstance(df, pd.DataFrame):
        print(f"[warning] '{name}' not found or not a DataFrame; skipping.")
        continue

    to_drop = _unnamed_cols(df)
    if to_drop:
        globals()[name] = df.drop(columns=to_drop)
        print(f"{name}: dropped {len(to_drop)} unnamed columns -> {to_drop}")
    else:
        print(f"{name}: no unnamed columns found.")


# Data Overview: schema and Sanity check
Verifying that each dataset loaded matches the structure (“schema”) as expected

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Normalize SINH_ID naming across all datasets and verify primary keys
# per-DataFrame (no mixing). For frames with SINH_ID (in any case/spacing form),
# rename to 'SINH_ID' and test PK (nulls + duplicates). For frames that lack
# SINH_ID (e.g., sdo1_distance, sdo4_dsa), search for a composite key (pairs,
# optionally triples) and report findings with diagnostics.
# -----------------------------------------------------------------------------

# --- 1) Rebuild registry safely (no mixing) ---
clean_names = list(custom_names.keys())
df_registry = {}
for name in clean_names:
    df = globals().get(name, None)
    if isinstance(df, pd.DataFrame):
        df_registry[name] = df
    else:
        print(f"[warning] DataFrame '{name}' not found. Skipping.")

# --- 2) Helpers ---
def _canonicalize(s: str) -> str:
    """Lowercase and remove non-alphanumerics for fuzzy header matching."""
    return re.sub(r'[^a-z0-9]', '', s.lower()) if isinstance(s, str) else s

def find_sinh_id_column(cols) -> str | None:
    """
    Detect a column that semantically matches SINH_ID even if the header's case,
    spacing or punctuation varies, e.g., 'sinh_id', 'Sinh Id', 'SINH-ID'.
    """
    want = "sinhid"
    for c in cols:
        if _canonicalize(c) == want:
            return c
    return None

def robust_min_max(s: pd.Series):
    if is_numeric_dtype(s):
        c = pd.to_numeric(s, errors="coerce")
        return c.min(), c.max()
    if is_datetime64_any_dtype(s):
        c = pd.to_datetime(s, errors="coerce")
        return c.min(), c.max()
    return None, None

def search_composite_key(df: pd.DataFrame, try_triples: bool = False, max_combos: int = 5000):
    """
    Attempt to find a valid composite key (pairs, optionally triples).
    Returns (candidate_cols: tuple|None, null_rows, dup_rows).
    """
    tested = 0

    # Pairs first
    for cols in combinations(df.columns, 2):
        if tested >= max_combos:
            break
        tested += 1
        sub = df[list(cols)]
        null_rows = int(sub.isna().any(axis=1).sum())
        dup_rows = int(df.duplicated(subset=list(cols)).sum())
        if null_rows == 0 and dup_rows == 0:
            return cols, null_rows, dup_rows

    if not try_triples:
        return None, None, None

    # Triples (can be heavy)
    for cols in combinations(df.columns, 3):
        if tested >= max_combos:
            break
        tested += 1
        sub = df[list(cols)]
        null_rows = int(sub.isna().any(axis=1).sum())
        dup_rows = int(df.duplicated(subset=list(cols)).sum())
        if null_rows == 0 and dup_rows == 0:
            return cols, null_rows, dup_rows

    return None, None, None

# --- 3) Normalize SINH_ID headers in-place, then test PKs ---
rename_log = []
pk_rows = []
dup_samples = {}  # dataset -> small df of duplicated key values (if any)

for ds_name, df in df_registry.items():
    # 3a) Normalize header to 'SINH_ID' if a fuzzy match exists
    match = find_sinh_id_column(df.columns)
    if match and match != "SINH_ID":
        df.rename(columns={match: "SINH_ID"}, inplace=True)
        rename_log.append((ds_name, match, "SINH_ID"))

    # 3b) Test PKs
    if "SINH_ID" in df.columns:
        nulls = int(df["SINH_ID"].isna().sum())
        dup_rows = int(df.duplicated(subset=["SINH_ID"]).sum())
        is_valid = (nulls == 0) and (dup_rows == 0)

        pk_rows.append({
            "dataset": ds_name,
            "has_SINH_ID": True,
            "pk_type": "single",
            "pk_columns": ("SINH_ID",),
            "null_rows": nulls,
            "dup_rows": dup_rows,
            "is_valid_pk": bool(is_valid),
        })

        # If invalid, capture a small sample of duplicates for inspection
        if not is_valid and dup_rows > 0:
            dupe_vals = (
                df["SINH_ID"]
                .value_counts(dropna=False)
                .loc[lambda s: s > 1]
                .head(10)
                .rename("count")
                .reset_index()
                .rename(columns={"index": "SINH_ID"})
            )
            dup_samples[ds_name] = dupe_vals

    else:
        # No SINH_ID present (e.g., sdo1_distance, sdo4_dsa) — search for composite
        cand_cols, nulls, dup_rows = search_composite_key(df, try_triples=False, max_combos=5000)
        is_valid = cand_cols is not None

        pk_rows.append({
            "dataset": ds_name,
            "has_SINH_ID": False,
            "pk_type": ("pair" if cand_cols and len(cand_cols) == 2 else
                        "triple" if cand_cols and len(cand_cols) == 3 else None),
            "pk_columns": cand_cols,
            "null_rows": int(nulls) if nulls is not None else None,
            "dup_rows": int(dup_rows) if dup_rows is not None else None,
            "is_valid_pk": bool(is_valid),
        })

# --- 4) Assemble reports ---
pk_report = (
    pd.DataFrame(pk_rows)
    .sort_values(["has_SINH_ID", "dataset"], ascending=[False, True])
    .reset_index(drop=True)
)

# Optional: a quick, filtered view of what you likely want to confirm
expected_pk_view = pk_report.loc[pk_report["has_SINH_ID"], ["dataset", "pk_columns", "null_rows", "dup_rows", "is_valid_pk"]]

# Emit logs so you can see what got renamed and any dup samples available
if rename_log:
    print("Renamed headers to normalize SINH_ID:")
    for ds, old, new in rename_log:
        print(f"  - {ds}: '{old}' -> '{new}'")

print("\nPrimary key validation summary (single SINH_ID where present; composite search otherwise):")
display(pk_report)

# If you want to quickly inspect duplicate SINH_IDs:
for ds_name, dup_df in dup_samples.items():
    print(f"\nTop duplicated SINH_IDs in {ds_name}:")
    display(dup_df)

# Example: enforce index for frames with valid single-column PK
for ds_name, df in df_registry.items():
    if "SINH_ID" in df.columns:
        if df["SINH_ID"].isna().sum() == 0 and df.duplicated(subset=["SINH_ID"]).sum() == 0:
            try:
                df.set_index("SINH_ID", inplace=True, drop=False)  # keep column + index
                print(f"Index set to SINH_ID for {ds_name}")
            except Exception as e:
                print(f"[warning] Could not set index for {ds_name}: {e}")


# Based on the above results, handle tables with invalid primary key:
sdo1_previous,
sdo2_orientation

-------------------------------------------------- sdo1_previous ----------------------------------------------------

In [ ]:
sdo1_previous

In [ ]:
sdo1_previous.columns

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Make SINH_ID the primary key in sdo1_previous by keeping only the row
# with the LATEST 'Final Exam Date' per SINH_ID. Rows with missing SINH_ID are dropped.
# Dates are parsed day-first (e.g., 28-06-2017). NaT dates are never selected when a
# real date exists for that SINH_ID.
# -----------------------------------------------------------------------------

df = sdo1_previous.copy()

# Parse dates (European format) and standardize SINH_ID
df["Final Exam Date"] = pd.to_datetime(df["Final Exam Date"], dayfirst=True, errors="coerce")
df["SINH_ID"] = pd.to_numeric(df["SINH_ID"], errors="coerce").round().astype("Int64")

# Drop rows without SINH_ID (cannot be a PK)
df = df.dropna(subset=["SINH_ID"]).copy()

# Sort so real dates come after NaT; keeping 'last' selects the latest real date
df = df.sort_values(["SINH_ID", "Final Exam Date"], ascending=[True, True], na_position="first")

# Keep only the latest row per SINH_ID
sdo1_previous = df.drop_duplicates(subset=["SINH_ID"], keep="last").copy()

# Sanity checks
assert sdo1_previous["SINH_ID"].isna().sum() == 0, "SINH_ID still has NaNs after filtering."
assert sdo1_previous.duplicated(subset=["SINH_ID"]).sum() == 0, "SINH_ID not unique after deduping."

print("sdo1_previous now unique on SINH_ID with latest Final Exam Date per student.")
print("Rows:", len(sdo1_previous))


In [ ]:
sdo1_previous

------------------------------------------ sdo2_orientation ------------------------------------------

In [ ]:
sdo2_orientation

In [ ]:
# For the sdo2_orientation data frame, check how many NaN are there in the SINH_ID column

nan_count = int(sdo2_orientation["SINH_ID"].isna().sum())
print(nan_count)

In [ ]:
# Check the count of rows with a NaN in at least two columns in the data frame.

count_at_least_two_nan = int((sdo2_orientation.isna().sum(axis=1) >= 2).sum())
print(count_at_least_two_nan)


In [ ]:

# Check the count of rows with a NaN in at least three columns in the data frame.

count_at_least_two_nan = int((sdo2_orientation.isna().sum(axis=1) >= 3).sum())
print(count_at_least_two_nan)


In [ ]:
# NaN count per column in sdo2_orientation
nan_sums = sdo2_orientation.isna().sum().astype(int).sort_values(ascending=False)
print(nan_sums)

In [ ]:
print(sdo2_orientation["Event_Types_Attended"].value_counts(dropna=False))

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Aggregate sdo2_orientation by SINH_ID, summing attendance metrics.
# Reducing to number of columns to three (most needed): SINH_ID, Number_of_Events_Attended, Number_of_Event_Types.
# -----------------------------------------------------------------------------

cols = ["SINH_ID", "Number_of_Events_Attended", "Number_of_Event_Types"]
df = sdo2_orientation[cols].copy()

# Ensure numeric for safe summation
df["Number_of_Events_Attended"] = pd.to_numeric(df["Number_of_Events_Attended"], errors="coerce")
df["Number_of_Event_Types"] = pd.to_numeric(df["Number_of_Event_Types"], errors="coerce")

sdo2_orientation = (
    df.groupby("SINH_ID", as_index=False)
      .agg(
          Number_of_Events_Attended=("Number_of_Events_Attended", "sum"),
          Number_of_Event_Types=("Number_of_Event_Types", "sum"),
      )
)

# Optional: preview
sdo2_orientation


// Some rows do not have SINH_ID. Merging needs to be on SINH_ID, or -if not available- on STUDENTNUMBER + Event_Date. We assume that a student attends an event prior to the next collegeyear (always starting on 1 Sept of a year. Such that if a student attends an event at 07-06-2021, the student can be coupled if the student is starting a degree in collegeyear 2021. 

In [ ]:
print(sdo2_orientation["Number_of_Events_Attended"].value_counts(dropna=False))

In [ ]:
print(sdo2_orientation["Number_of_Event_Types"].value_counts(dropna=False))

In [ ]:
# NaN count per column in sdo2_orientation
nan_sums = sdo2_orientation.isna().sum().astype(int).sort_values(ascending=False)
print(nan_sums)

---------------------- Normalize the names of the columns for each dataframe-----------------------------------

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Standardize column names and add the DATAFRAME NAME as a PREFIX
# (provenance) in an *idempotent* way. Re-running this cell will not double-prefix.
#
# Steps per DataFrame:
#   1) Convert column names to underscored form (e.g., "Final Exam Date" -> "Final_Exam_Date").
#   2) For NON-key columns, ensure they are prefixed with "<dataset>_" (e.g., "sdo1_previous_Final_Exam_Date").
#   3) Do NOT change preserved join keys (default: 'SINH_ID').
#   4) Avoid double changes:
#        • If a column already has the desired "<dataset>_" prefix, leave it.
#        • If it has a trailing "_<dataset>" suffix from earlier steps, strip it before adding the prefix.
#   5) Ensure final names are unique (add __2, __3 if collisions occur).
# -----------------------------------------------------------------------------


df_names = [
    "sdo1_distance",
    "sdo1_characteristics",
    "sdo1_previous",
    "sdo2_skc",
    "sdo2_orientation",
    "sdo4_dsa",
    "sdo5_degree",
    "sdo6_results",
    "sdo7_questionaire"
]

# Join keys to KEEP as-is (no prefix)
preserve_keys = {"SINH_ID"}  # add more if needed, e.g., {"SINH_ID", "POSTCODE"}

def underscore_words(name: str) -> str:
    s = str(name)
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s)        # camelCase -> camel_Case
    s = re.sub(r"([A-Za-z])([0-9])", r"\1_\2", s)        # A1 -> A_1
    s = re.sub(r"([0-9])([A-Za-z])", r"\1_\2", s)        # 1A -> 1_A
    s = re.sub(r"[^0-9A-Za-z]+", "_", s)                 # spaces/punct -> _
    s = re.sub(r"_+", "_", s).strip("_")                 # collapse/trim _
    return s

# Normalize preserved keys for comparisons after underscoring
preserve_keys_norm = {underscore_words(k) for k in preserve_keys}

for name in df_names:
    df = globals().get(name)
    if not isinstance(df, pd.DataFrame):
        print(f"[warning] {name} not found or not a DataFrame; skipping.")
        continue

    prefix = f"{name}_"
    suffix = f"_{name}"

    current_cols = list(df.columns)

    # If ALL non-key columns already start with "<dataset>_", we consider it done (idempotent).
    non_key_cols = [c for c in current_cols if underscore_words(c) not in preserve_keys_norm]
    if non_key_cols and all(str(c).startswith(prefix) for c in non_key_cols):
        print(f"{name}: already prefixed; no changes made.")
        continue

    col_map = {}
    for orig in current_cols:
        base = underscore_words(orig)

        # If previous step added a suffix (e.g., base_<dataset>), strip it first
        if base.endswith(suffix):
            base = base[: -len(suffix)]

        # If the desired prefix is already present, leave it
        if base.startswith(prefix):
            target = base

        # Preserve join keys exactly (no prefix)
        elif base in preserve_keys_norm:
            target = base
            # If someone previously prefixed/suffixed a key, leave as-is to avoid breaking joins
            if str(orig).startswith(prefix) or str(orig).endswith(suffix):
                # no rename for keys that already carry dataset tags
                continue

        else:
            # Add the dataset prefix
            target = f"{prefix}{base}"

        if orig != target:
            col_map[orig] = target

    if not col_map:
        print(f"{name}: no columns needed renaming; skipped.")
        continue

    # Ensure uniqueness of targets (disambiguate collisions if any)
    proposed = list(col_map.values())
    counts = Counter(proposed)
    if any(cnt > 1 for cnt in counts.values()):
        seen = defaultdict(int)
        for k, v in list(col_map.items()):
            seen[v] += 1
            if counts[v] > 1:
                if seen[v] == 1:
                    continue  # first keeps base
                col_map[k] = f"{v}__{seen[v]}"

    globals()[name] = df.rename(columns=col_map)
    print(f"{name}: renamed {len(col_map)} column(s); keys preserved: {sorted(preserve_keys)}")


In [ ]:
sdo1_distance

In [ ]:

df_names = [
    "sdo1_distance",
    "sdo1_characteristics",
    "sdo1_previous",
    "sdo2_skc",
    "sdo2_orientation",
    "sdo4_dsa",
    "sdo5_degree",
    "sdo6_results",
    "sdo7_questionaire",
]

summary = []
for name in df_names:
    df = globals().get(name)
    if isinstance(df, pd.DataFrame):
        r, c = df.shape
        print(f"{name}: {r} rows × {c} columns")
        summary.append({"dataset": name, "rows": r, "cols": c})
    else:
        print(f"[warning] {name} not found or not a DataFrame")

# Optional compact table (comment out if you only want the prints)
shape_report = pd.DataFrame(summary).sort_values("dataset").reset_index(drop=True)
shape_report


# Redo schema and Sanity check after handling duplicate SINH_ID

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Normalize SINH_ID naming across all datasets and verify primary keys
# per-DataFrame (no mixing). For frames with SINH_ID (in any case/spacing form),
# rename to 'SINH_ID' and test PK (nulls + duplicates). For frames that lack
# SINH_ID (e.g., sdo1_distance, sdo4_dsa), search for a composite key (pairs,
# optionally triples) and report findings with diagnostics.
# -----------------------------------------------------------------------------

# --- 1) Rebuild registry safely (no mixing) ---
clean_names = list(custom_names.keys())
df_registry = {}
for name in clean_names:
    df = globals().get(name, None)
    if isinstance(df, pd.DataFrame):
        df_registry[name] = df
    else:
        print(f"[warning] DataFrame '{name}' not found. Skipping.")

# --- 2) Helpers ---
def _canonicalize(s: str) -> str:
    """Lowercase and remove non-alphanumerics for fuzzy header matching."""
    return re.sub(r'[^a-z0-9]', '', s.lower()) if isinstance(s, str) else s

def find_sinh_id_column(cols) -> str | None:
    """
    Detect a column that semantically matches SINH_ID even if the header's case,
    spacing or punctuation varies, e.g., 'sinh_id', 'Sinh Id', 'SINH-ID'.
    """
    want = "sinhid"
    for c in cols:
        if _canonicalize(c) == want:
            return c
    return None

def robust_min_max(s: pd.Series):
    if is_numeric_dtype(s):
        c = pd.to_numeric(s, errors="coerce")
        return c.min(), c.max()
    if is_datetime64_any_dtype(s):
        c = pd.to_datetime(s, errors="coerce")
        return c.min(), c.max()
    return None, None


# // No need for this: key knowledge is functional. This function also does not seem to find the proper (composite) keys. 
def search_composite_key(df: pd.DataFrame, try_triples: bool = False, max_combos: int = 5000):
    """
    Attempt to find a valid composite key (pairs, optionally triples).
    Returns (candidate_cols: tuple|None, null_rows, dup_rows).
    """
    tested = 0

    # Pairs first
    for cols in combinations(df.columns, 2):
        if tested >= max_combos:
            break
        tested += 1
        sub = df[list(cols)]
        null_rows = int(sub.isna().any(axis=1).sum())
        dup_rows = int(df.duplicated(subset=list(cols)).sum())
        if null_rows == 0 and dup_rows == 0:
            return cols, null_rows, dup_rows

    if not try_triples:
        return None, None, None

    # Triples (can be heavy)
    for cols in combinations(df.columns, 3):
        if tested >= max_combos:
            break
        tested += 1
        sub = df[list(cols)]
        null_rows = int(sub.isna().any(axis=1).sum())
        dup_rows = int(df.duplicated(subset=list(cols)).sum())
        if null_rows == 0 and dup_rows == 0:
            return cols, null_rows, dup_rows

    return None, None, None

# --- 3) Normalize SINH_ID headers in-place, then test PKs ---
rename_log = []
pk_rows = []
dup_samples = {}  # dataset -> small df of duplicated key values (if any)

# Add a 'rows' column (rowcount per DataFrame) and show it right after 'dataset'

# ... keep everything above unchanged ...

for ds_name, df in df_registry.items():
    # 3a) Normalize header to 'SINH_ID' if a fuzzy match exists
    match = find_sinh_id_column(df.columns)
    if match and match != "SINH_ID":
        df.rename(columns={match: "SINH_ID"}, inplace=True)
        rename_log.append((ds_name, match, "SINH_ID"))

    # 3b) Test PKs
    n_rows = int(len(df))  # <-- rowcount

    if "SINH_ID" in df.columns:
        nulls = int(df["SINH_ID"].isna().sum())
        dup_rows = int(df.duplicated(subset=["SINH_ID"]).sum())
        is_valid = (nulls == 0) and (dup_rows == 0)

        pk_rows.append({
            "dataset": ds_name,
            "rows": n_rows,                 # <-- add here
            "has_SINH_ID": True,
            "pk_type": "single",
            "pk_columns": ("SINH_ID",),
            "null_rows": nulls,
            "dup_rows": dup_rows,
            "is_valid_pk": bool(is_valid),
        })

        if not is_valid and dup_rows > 0:
            dupe_vals = (
                df["SINH_ID"]
                .value_counts(dropna=False)
                .loc[lambda s: s > 1]
                .head(10)
                .rename("count")
                .reset_index()
                .rename(columns={"index": "SINH_ID"})
            )
            dup_samples[ds_name] = dupe_vals

    else:
        cand_cols, nulls, dup_rows = search_composite_key(df, try_triples=False, max_combos=5000)
        is_valid = cand_cols is not None

        pk_rows.append({
            "dataset": ds_name,
            "rows": n_rows,                 # <-- add here
            "has_SINH_ID": False,
            "pk_type": ("pair" if cand_cols and len(cand_cols) == 2 else
                        "triple" if cand_cols and len(cand_cols) == 3 else None),
            "pk_columns": cand_cols,
            "null_rows": int(nulls) if nulls is not None else None,
            "dup_rows": int(dup_rows) if dup_rows is not None else None,
            "is_valid_pk": bool(is_valid),
        })

# --- 4) Assemble reports (show 'rows' right after 'dataset') ---
pk_report = (
    pd.DataFrame(pk_rows)
      .sort_values(["has_SINH_ID", "dataset"], ascending=[False, True])
      .reset_index(drop=True)
)

# Reorder columns for display: dataset, rows, then the rest
cols_order = ["dataset", "rows", "has_SINH_ID", "pk_type", "pk_columns", "null_rows", "dup_rows", "is_valid_pk"]
pk_report = pk_report.loc[:, cols_order]

print("\nPrimary key validation summary (single SINH_ID where present; composite search otherwise):")
display(pk_report)


Decision: 
Ignore the sdo7_questionaire dataframe , it looks very messy.

// Major data wrangling is needed to combine sdo7 with other tables. 

In [ ]:
# Remove sdo7_questionaire DataFrame from the global namespace
globals().pop("sdo7_questionaire", None)  # safe even if it doesn't exist

# Optional: free memory aggressively
import gc; gc.collect()

# Combining data frames. 
Data frames combination strategy: 
1. Use SINH_ID for all tables with this primary key, ref SCHEMA & Sanity check
2. For tables without the primary key, use results from "the top 3 matches per dataset pair" and "Sanity check" to decide.

// Can you change the term 'cohort' to something else? Cohort is also a functional term within the HU and within this project! It might become confusing


--------------------- 1. Use SINH_ID for all tables with this primary key ----------------------------



In [ ]:
sdo1_previous.SINH_ID.nunique()

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Keep ALL sdo5_degree columns as the base and LEFT-join the other
# Reason: sdo5_degree has the highest number of stable observations, less data cutdown
# student tables on SINH_ID — with **no suffixes** added. We first assert that
# there are no overlapping column names (except the join key) so pandas won’t
# auto-create _x/_y. If overlap exists, the merge will raise with a clear error.
# -----------------------------------------------------------------------------

def ensure_plain_column_key(df, key="SINH_ID"):
    df = df.copy()
    if isinstance(df.index, pd.MultiIndex):
        if key in (df.index.names or []):
            df = df.reset_index(drop=(key in df.columns))
    elif df.index.name == key:
        df = df.reset_index(drop=(key in df.columns))
    return df.rename_axis(None)

def as_int64(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").round().astype("Int64")

def one_row_per_student(df, date_col=None, dayfirst=True):
    df = ensure_plain_column_key(df, "SINH_ID").copy()
    df["SINH_ID"] = as_int64(df["SINH_ID"])
    if date_col and date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce", dayfirst=dayfirst)
        df = df.sort_values(["SINH_ID", date_col]).drop_duplicates(["SINH_ID"], keep="last")
    else:
        df = df.drop_duplicates(["SINH_ID"], keep="last")
    return df

def assert_no_overlap(left: pd.DataFrame, right: pd.DataFrame, key: str = "SINH_ID"):
    overlap = (set(left.columns) & set(right.columns)) - {key}
    if overlap:
        raise ValueError(
            f"Refusing to merge because these columns would collide without suffixes: {sorted(overlap)}.\n"
            f"Rename/drop before merging, or allow suffixes."
        )

def left_merge_no_suffix(left: pd.DataFrame, right: pd.DataFrame, on: str = "SINH_ID"):
    assert_no_overlap(left, right, on)
    before = len(left)
    out = left.merge(right, on=on, how="left", validate="one_to_one")
    assert len(out) == before, f"Row count changed on merge with key '{on}'"
    return out

# ---- Base population (KEEP all sdo5_degree columns) ----
cohort = ensure_plain_column_key(sdo5_degree, "SINH_ID").copy()
cohort["SINH_ID"] = as_int64(cohort["SINH_ID"])
# If needed: ensure unique per SINH_ID
# cohort = cohort.sort_values("SINH_ID").drop_duplicates("SINH_ID", keep="last")
base_n = len(cohort)

# ---- Prepare sources as 1:1 on SINH_ID ----
char    = one_row_per_student(sdo1_characteristics)
prev    = one_row_per_student(sdo1_previous, date_col="Final Exam Date", dayfirst=True)
skc     = one_row_per_student(sdo2_skc,      date_col="SKC_DATUM",      dayfirst=True) if "SKC_DATUM" in sdo2_skc.columns else one_row_per_student(sdo2_skc)
results = one_row_per_student(sdo6_results)

# Orientation aggregated to 1 row per SINH_ID
orient = ensure_plain_column_key(sdo2_orientation, "SINH_ID").copy()
orient["SINH_ID"] = as_int64(orient["SINH_ID"])
for c in ["Number_of_Events_Attended", "Number_of_Event_Types"]:
    orient[c] = pd.to_numeric(orient.get(c, 0), errors="coerce")
orient = (orient.groupby("SINH_ID", as_index=False)
                .agg(Number_of_Events_Attended=("Number_of_Events_Attended","sum"),
                     Number_of_Event_Types=("Number_of_Event_Types","sum")))

# ---- Presence flags (computed before merging) ----
has_char    = set(char["SINH_ID"].dropna())
has_prev    = set(prev["SINH_ID"].dropna())
has_skc     = set(skc["SINH_ID"].dropna())
has_results = set(results["SINH_ID"].dropna())
has_orient  = set(orient["SINH_ID"].dropna())

# ---- Merge (LEFT from full base) — NO SUFFIXES ----
cohort = left_merge_no_suffix(cohort, char,    on="SINH_ID")
cohort = left_merge_no_suffix(cohort, prev,    on="SINH_ID")
cohort = left_merge_no_suffix(cohort, skc,     on="SINH_ID")
cohort = left_merge_no_suffix(cohort, results, on="SINH_ID")
cohort = left_merge_no_suffix(cohort, orient,  on="SINH_ID")

# ---- Add presence flags ----
cohort["has_characteristics"] = cohort["SINH_ID"].isin(has_char).astype("Int8")
cohort["has_previous"]        = cohort["SINH_ID"].isin(has_prev).astype("Int8")
cohort["has_skc"]             = cohort["SINH_ID"].isin(has_skc).astype("Int8")
cohort["has_results"]         = cohort["SINH_ID"].isin(has_results).astype("Int8")
cohort["has_orientation"]     = cohort["SINH_ID"].isin(has_orient).astype("Int8")

# ---- Safe default fills for orientation counts ----
for c in ["Number_of_Events_Attended", "Number_of_Event_Types"]:
    if c in cohort.columns:
        cohort[c] = cohort[c].fillna(0).astype("Int64")

print("Base students:", base_n)
print("After merges :", len(cohort))  # should match base


In [ ]:
# cohort is the new dataframe after this merge,
# later it will be merged with the non-primary key tables using composites
cohort

In [ ]:
cohort.columns

----------------------2. For tables without the primary key, use results from "the top 3 matches per dataset pair" and "Sanity check".--------------------- 

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Identify and evaluate non-SINH_ID columns that 
# can serve as reliable join keys across the datasets by 
# standardizing formats and testing uniqueness, overlap, and coverage.
# cohort dataframe = sdo1_characteristics + sdo1_previous + sdo2_orientation + sdo2_skc + sdo5_degree + sdo6_results
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# Top 3 matches per dataset pair ONLY, with columns ordered as:
# DF A | DF B | Column_pair | Name_sim | Score | Overlap
# -----------------------------------------------------------------------------


# ---- DataFrames to compare (must already exist as globals) ----
df_names = [
    "sdo1_distance",
    "sdo4_dsa",
    "cohort"
]
df_registry = {n: globals().get(n) for n in df_names if isinstance(globals().get(n), pd.DataFrame)}

# ---- If similar_pairs not already computed, build it quickly ----
if "similar_pairs" not in globals():
    NAME_THR, VALUE_THR = 0.60, 0.60
    MAX_UNIQUE = 10_000
    RNG = np.random.default_rng(42)

    def name_similarity(a: str, b: str) -> float:
        toks = lambda s: set(re.findall(r"[a-z0-9]+", str(s).lower()))
        A, B = toks(a), toks(b)
        if not A and not B: return 0.0
        j = len(A & B) / len(A | B) if (A | B) else 0.0
        na = re.sub(r"[^a-z0-9]", "", str(a).lower())
        nb = re.sub(r"[^a-z0-9]", "", str(b).lower())
        return 1.0 if na and na == nb else j

    def std_uniques(s: pd.Series) -> set:
        s_num = pd.to_numeric(s, errors="coerce")
        if s_num.notna().mean() > 0.9 or getattr(s.dtype, "kind", "") in "ifub":
            vals = s_num.dropna().unique()
            def fmt(x):
                xf = float(x)
                return str(int(xf)) if xf.is_integer() else str(xf)
            arr = np.array([fmt(v) for v in vals], dtype=object)
        else:
            arr = s.astype(str).str.strip().str.lower()
            arr = arr[arr.ne("nan")].unique()
        if len(arr) > MAX_UNIQUE:
            arr = arr[RNG.choice(len(arr), size=MAX_UNIQUE, replace=False)]
        return set(arr.tolist())

    rows = []
    for (dfa_name, dfa), (dfb_name, dfb) in combinations(df_registry.items(), 2):
        for ca in dfa.columns:
            sa = dfa[ca]
            if sa.notna().sum() == 0:
                continue
            Au = std_uniques(sa)
            if not Au:
                continue
            for cb in dfb.columns:
                sb = dfb[cb]
                if sb.notna().sum() == 0:
                    continue
                Bu = std_uniques(sb)
                if not Bu:
                    continue
                inter = Au & Bu
                if not inter:
                    continue
                union = Au | Bu
                containment = len(inter) / min(len(Au), len(Bu))
                jacc = len(inter) / len(union)
                value_score = max(containment, jacc)
                ns = name_similarity(ca, cb)

                if (ns >= NAME_THR and value_score >= VALUE_THR) or value_score >= 0.90:
                    rows.append({
                        "df1": dfa_name, "col1": str(ca),
                        "df2": dfb_name, "col2": str(cb),
                        "name_sim": ns,
                        "value_containment": containment,
                        "value_jaccard": jacc,
                        "value_score": value_score,
                        "intersection": len(inter),
                        "unique_df1": len(Au),
                        "unique_df2": len(Bu),
                    })

    similar_pairs = (pd.DataFrame(rows)
                     .sort_values(["value_score", "name_sim", "intersection"], ascending=[False, False, False])
                     .reset_index(drop=True))

# ---- Top 3 per dataset pair, with requested column order ----
top_per_dfpair = (
    similar_pairs
    .sort_values(["value_score", "name_sim", "intersection"], ascending=[False, False, False])
    .groupby(["df1", "df2"], as_index=False)
    .head(3)
    .assign(
        Column_pair=lambda d: d["df1"] + "." + d["col1"] + "  ↔  " + d["df2"] + "." + d["col2"],
        Name_sim=lambda d: (d["name_sim"] * 100).round(1).astype(str) + "%",
        Score=lambda d: (d["value_score"] * 100).round(1).astype(str) + "%",
        Overlap=lambda d: d["intersection"].astype(int).astype(str)
                           + " / " + d[["unique_df1", "unique_df2"]].min(axis=1).astype(int).astype(str),
    )
    .loc[:, ["df1", "df2", "Column_pair", "Name_sim", "Score", "Overlap"]]
    .rename(columns={"df1": "DF A", "df2": "DF B"})
)

print("\nTop 3 matches per dataset pair:")
display(top_per_dfpair)



// Interesting approach. I can't say that I fully understand HOW it works, and also I'm not convinced it is fully valid: it misses one obvious match: sdo1_distance with cohort on sdo1_distance.sdo1_distance_postcode = cohort.sdo1_previous_Previous_School_Postal_Code. 
Other matches are spot-on!


Decision: 

Primary cross-table matches (excluding SINH_ID): 
Use academic year—sdo4_dsa_Collegejaar ↔ sdo5_degree_COLLEGEJAAR —after normalizing to a consistent year format; 
Use program identifiers—sdo4_dsa_CROHO code ↔ sdo5_degree.degree_code—for program-level joins; and 
Use postcode—sdo1_distance.postcode ↔ sdo1_previous.Previous School Postal Code—for geography lookups (standardize postcode formatting before joining). 
Ignore other apparent 100% pairs such as “id ↔ age_start_study”, event counts, or Dutch_nationality against binary flags, as these are small, low-cardinality coincidences rather than true keys.

Perfect (100%) value-overlap pairs: 
Collegejaar ↔ COLLEGEJAAR (5/5 years), CROHO code ↔ degree_code (23/23 codes), and postcode ↔ Previous School Postal Code (747/747 postcodes).

---------------------------------------Normalize Pairs of columns before final merge -------------------------------------
Normalize keys of selected columns below before merging, normalizing to make sure they have the same format.  

comment:  cohort data frame is resulting from earlier merging based on SINH_ID

sdo5_degree_COLLEGEJAAR (from cohort data frame) ↔ sdo4_dsa_Collegejaar (from sdo4_dsa data frame)
sdo5_degree_degree_code (from cohort data frame) ↔ sdo4_dsa_CROHO_code (from sdo4_dsa data frame)
sdo5_degree_POSTCODE (from cohort data frame) ↔ sdo1_distance_postcode (from sdo1_distance data frame)

-------------- sdo5_degree_COLLEGEJAAR (from cohort data frame) ↔ sdo4_dsa_Collegejaar (from sdo4_dsa data frame)----------------

In [ ]:
print(cohort["sdo5_degree_COLLEGEJAAR"].value_counts(dropna=False))

In [ ]:
print(sdo4_dsa["sdo4_dsa_Collegejaar"].value_counts(dropna=False))

Decision: Both are int64 same data types and same format.

------------------ sdo5_degree_degree_code (from cohort data frame) ↔ sdo4_dsa_CROHO_code (from sdo4_dsa data frame) ---------- 

In [ ]:
print(cohort["sdo5_degree_degree_code"].value_counts(dropna=False))

In [ ]:
print(sdo4_dsa["sdo4_dsa_CROHO_code"].value_counts(dropna=False))

Decision: Both are int64 same data type and same format 

--------------sdo5_degree_POSTCODE (from cohort data frame) ↔ sdo1_distance_postcode (from sdo1_distance data frame)------------

In [ ]:
print(cohort["sdo5_degree_POSTCODE"].value_counts(dropna=False))

In [ ]:
print(sdo1_distance["sdo1_distance_postcode"].value_counts(dropna=False))

Decision: Both are int64 same data type but there is need to remove the decimals from sdo5_degree_POSTCODE

In [ ]:
# Make sure 'sdo5_degree_POSTCODE' has no decimal part, keeping a numeric dtype.
# If it's already integer-typed, nothing changes. If it's float/object, coerce to
# numeric and cast to pandas' nullable integer (Int64) to preserve NaNs.
# NOTE: If  postcodes  have leading zeros, casting to int will drop them.

col = "sdo5_degree_POSTCODE"
if col not in cohort.columns:
    raise KeyError(f"Column '{col}' not found in cohort.")

s = cohort[col]

if pd.api.types.is_integer_dtype(s):
    # Already integer (no decimals possible)
    pass
else:
    s_num = pd.to_numeric(s, errors="coerce")
    # If any values have a fractional part (not just .0), round to nearest int
    frac_part = np.modf(s_num.fillna(0))[0]
    if (np.abs(frac_part) > 0).any():
        s_num = s_num.round(0)
    cohort[col] = s_num.astype("Int64")  # nullable integer (keeps NaN)

In [ ]:
# confirm results
print(cohort["sdo5_degree_POSTCODE"].value_counts(dropna=False))

# Final merge

In [ ]:
# -----------------------------------------------------------------------------
# Merge plan (NO suffixes): cohort  ⟵ sdo4_dsa on (program, year)  and  ⟵ sdo1_distance on postcode
# Joins:
#   cohort.sdo5_degree_COLLEGEJAAR ↔ sdo4_dsa.sdo4_dsa_Collegejaar
#   cohort.sdo5_degree_degree_code ↔ sdo4_dsa.sdo4_dsa_CROHO_code
#   cohort.sdo5_degree_POSTCODE    ↔ sdo1_distance.sdo1_distance_postcode
# -----------------------------------------------------------------------------


# --- helpers ---
def norm_year(s: pd.Series) -> pd.Series:
    return s.astype(str).str.extract(r"(\d{4})", expand=False)

def norm_prog_code(s: pd.Series) -> pd.Series:
    # keep digits only; we will zfill to a common width
    return s.astype(str).str.strip().str.extract(r"(\d+)", expand=False)

def norm_postcode(s: pd.Series) -> pd.Series:
    return s.astype(str).str.replace(r"\s+", "", regex=True).str.upper()

def assert_no_overlap(left: pd.DataFrame, right: pd.DataFrame, ignore: set[str]):
    overlap = (set(left.columns) & set(right.columns)) - set(ignore)
    if overlap:
        raise ValueError(
            "Column name collision without suffixes: "
            f"{sorted(overlap)}. Rename/drop before merging or allow suffixes."
        )

# --- 0) Make working copies
co = cohort.copy()
dsa = sdo4_dsa.copy()
dist = sdo1_distance.copy()

# --- 1) Normalize join keys in each frame
# cohort keys
co_year_col  = "sdo5_degree_COLLEGEJAAR"
co_prog_col  = "sdo5_degree_degree_code"
co_post_col  = "sdo5_degree_POSTCODE"

if not {co_year_col, co_prog_col}.issubset(co.columns):
    raise KeyError("cohort must contain 'sdo5_degree_COLLEGEJAAR' and 'sdo5_degree_degree_code'.")

co["__YEAR__"]  = norm_year(co[co_year_col])
co["__PROG__"]  = norm_prog_code(co[co_prog_col])
co["__PCODE__"] = norm_postcode(co[co_post_col]) if co_post_col in co.columns else pd.NA

# sdo4_dsa keys
dsa_year_col = "sdo4_dsa_Collegejaar"
dsa_prog_col = "sdo4_dsa_CROHO_code"
if not {dsa_year_col, dsa_prog_col}.issubset(dsa.columns):
    raise KeyError("sdo4_dsa must contain 'sdo4_dsa_Collegejaar' and 'sdo4_dsa_CROHO_code'.")

dsa["__YEAR__"] = norm_year(dsa[dsa_year_col])
dsa["__PROG__"] = norm_prog_code(dsa[dsa_prog_col])

# Align zero-padding width for program codes
width = int(pd.concat([
    co["__PROG__"].dropna().map(len),
    dsa["__PROG__"].dropna().map(len)
], ignore_index=True).max() or 0)
if width > 0:
    co["__PROG__"]  = co["__PROG__"].str.zfill(width)
    dsa["__PROG__"] = dsa["__PROG__"].str.zfill(width)

# sdo1_distance key
dist_pc_col = "sdo1_distance_postcode"
if dist_pc_col not in dist.columns:
    raise KeyError("sdo1_distance must contain 'sdo1_distance_postcode'.")
dist["__PCODE__"] = norm_postcode(dist[dist_pc_col])

# --- 2) Deduplicate right-hand tables to their keys (dimension style)
dsa_dim = (dsa.sort_values(["__PROG__", "__YEAR__"])
             .drop_duplicates(subset=["__PROG__", "__YEAR__"], keep="first"))

dist_dim = (dist.sort_values("__PCODE__")
              .drop_duplicates(subset=["__PCODE__"], keep="first"))

# --- 3) Diagnostics (optional but recommended)
prog_year_students = co[["__PROG__", "__YEAR__"]].dropna().drop_duplicates()
prog_year_dsa      = dsa_dim[["__PROG__", "__YEAR__"]].dropna().drop_duplicates()
print(f"PROGRAM/YEAR keys — cohort: {len(prog_year_students)}, dsa: {len(prog_year_dsa)}, "
      f"intersection: {len(prog_year_students.merge(prog_year_dsa, on=['__PROG__','__YEAR__']))}")

if co_post_col in co.columns:
    pc_students = co["__PCODE__"].dropna().unique()
    pc_distinct = dist_dim["__PCODE__"].dropna().unique()
    inter_pc = np.intersect1d(pc_students, pc_distinct)
    print(f"POSTCODE keys — cohort: {len(pc_students)}, distance: {len(pc_distinct)}, intersection: {len(inter_pc)}")

# --- 4) Merges (LEFT from cohort), NO SUFFIXES, with m:1 validation
# DSA merge
assert_no_overlap(co, dsa_dim, ignore={"__PROG__", "__YEAR__"})
rows_before = len(co)
co = co.merge(dsa_dim, on=["__PROG__", "__YEAR__"], how="left", validate="m:1")
assert len(co) == rows_before, "Row count changed after DSA merge."

# Distance merge
assert_no_overlap(co, dist_dim, ignore={"__PCODE__"})
rows_before = len(co)
co = co.merge(dist_dim, on="__PCODE__", how="left", validate="m:1")
assert len(co) == rows_before, "Row count changed after distance merge."

# --- 5) (Optional) drop helper keys
co.drop(columns=["__PROG__", "__YEAR__", "__PCODE__"], inplace=True, errors="ignore")

# Final merged DataFrame (keeps ALL cohort columns)
data = co
data

In [ ]:
data.columns

In [ ]:
# // sdo2 does not seem to be merged well: columns Number_of_Events_attend and Number_of_Event_Types = 0. 
print(data["Number_of_Events_Attended"].sum(min_count=1))
print(data["Number_of_Event_Types"].sum(min_count=1))

In [ ]:
# NaN count per column in students
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False)
print(nan_sums)

Decision: 
The NaN counts is not bad, anything below 12000 NaN can be imputed, this is not more than 50% of missing values.
For those >40,000 missing values, that is close to the total data size 47,000, this will be decided later

// For the DSA column, please review the notes on how this data should be used (in the data folder). 

// For the other columns > 40K: these are not necessary and can be deleted. 

In [ ]:
data.SINH_ID.nunique()

----------------- Save combined data frame before further cleaning -----------------------

In [ ]:
OUT_DIR = Path(os.getenv("OUTPUTS_DIR")).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / "v2_combined.csv"
data.to_csv(out_path, index=False)

print("Wrote:", out_path)
# upload it to sharepoint and remove data from processed folder.

# TO BE CONTINUED

In [ ]:
# Drop unnecessary columns from the combined DataFrame data
data = data.drop(
    columns=[
        "sdo1_d_student_id",
        "sdo5_degree_code_letters",
        "sdo5_degree_code"
    ],
    errors="ignore"
)


In [ ]:
# Ensure no two or more columns have the same values
identical_columns = []
columns = data.columns.tolist()

for i in range(len(columns)):
    for j in range(i+1, len(columns)):
        col1, col2 = columns[i], columns[j]
        if data[col1].equals(data[col2]):
            identical_columns.append((col1, col2))

if identical_columns:
    print("Identical columns found:")
    for col_pair in identical_columns:
        print(col_pair)
else:
    print("No identical columns found.")

In [ ]:
# Check for columns with only one unique value
single_value_cols = [col for col in data.columns if data[col].nunique() == 1]

print("Columns with only one unique value:", single_value_cols)


In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
#pd.set_option('display.max_rows', None)   # Show all rows (be careful if huge!)
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_colwidth', None)
#pd.set_option('display.width', 0)         # Automatically adjusts line-wrapping
#pd.set_option('display.expand_frame_repr', False)  


In [ ]:
# Print value counts for all columns
for col in data.columns:
    print(f"Value counts for column: {col}")
    print(data[col].value_counts(dropna=False))  # include NaN values in the count
    print("\n" + "="*50 + "\n")

In [ ]:
data.columns

In [ ]:
# Convert date-related columns to datetime
date_cols = [
    "sdo1_date_of_birth",
    "sdo2_SKC_DATUM",
    "sdo5_enrollment_date",
    "sdo5_degree_start_date"
]

for col in date_cols:
    if col in data.columns:
        data[col] = pd.to_datetime(data[col], errors="coerce")  # invalid dates -> NaT


In [ ]:
# Print value counts only for object (categorical) columns, excluding skip_cols and datetime columns
# Columns to skip manually
skip_cols = ["SINH_ID", "sdo5_POSTCODE"]

for col in data.select_dtypes(include=["object"]).columns:
    if col in skip_cols or col in data.select_dtypes(include=["datetime"]).columns:
        continue
    print(f"Value counts for column: {col}")
    print(data[col].value_counts(dropna=False))  # include NaN values in the count
    print("\n" + "="*50 + "\n")


Questions: 

What is the difference between sdo1_nationality and sdo5_LAND columns
What does 0 in sdo1_gender column means

// Fraukje: LAND is the country that is registered as the student's postal address at the start of the study. I suppose it is not useful by itself, but maybe as an indicator of someone living abroad: has_foreign_address = 1 if LAND != The Netherlands else 0

In [ ]:
# Delete sdo5_LAND for now
del data['sdo5_LAND']

---------------------------------Handle categories in sdo1_nationality column ------------------------------------------------------
//I thought we agreed that we would not use Nationality as a category, but only as a bit indicating Dutch national yes/no?

In [ ]:
# Clean categorical column (sdo1_nationality):
# - Replace unknown codes ("XXX", "XX") with "Unknown"
# - Group all categories with <1% frequency into "Other"
# - Keep only dominant categories (e.g., "NL") intact


col = "sdo1_nationality"
threshold = 0.01  # 1% cutoff

# Step 1: explicitly map special unknown codes
data[col] = data[col].replace(["XXX", "XX"], "Unknown")

# Step 2: group rare categories into "Other"
freq = data[col].value_counts(normalize=True)
rare_cats = freq[freq < threshold].index
data[col] = data[col].replace(rare_cats, "Other")

# Quick check
print(data["sdo1_nationality"].value_counts())             # raw counts
print(data["sdo1_nationality"].value_counts(normalize=True))  # proportions


-------------------------------------- Handle categories in sdo2_SKC_RESULT Column -------------------------------------------

In [ ]:
# Clean categorical column (sdo2_SKC_RESULT):
# - Replace categories with fewer than 100 occurrences with "Other"
#// Can we keep even the small categories intact (there are only a small number of categories anyway)?

col = "sdo2_SKC_RESULT"
threshold = 100

# Find categories with counts below the threshold
value_counts = data[col].value_counts()
rare_cats = value_counts[value_counts < threshold].index

# Replace them with "Other"
data[col] = data[col].replace(rare_cats, "Other")

# Quick check
print(data[col].value_counts(dropna=False))

-------------------------------------- Handle categories in sdo2_ADVIES_DEF Column -------------------------------------------

In [ ]:
# Clean categorical values in sdo2_ADVIES_DEF:
# - Remove the word "met"
# - Replace spaces between remaining words with underscores
# - Preserve NaN values and original capitalization
#//Is it necessary to do cleaning? It makes the categories less interpretable for end users.
#//Can we keep even the small categories intact (there are only a small number of categories anyway)?

col = "sdo2_ADVIES_DEF"

# Apply cleaning only on non-null values
data[col] = data[col].dropna().apply(
    lambda x: "_".join([w for w in x.split() if w.lower() != "met"])
)

# Quick check
print(data[col].value_counts())


-------------------------------------- Handle categories in sdo5_degree Column -------------------------------------------

In [ ]:
# Clean categories in sdo5_degree:
# - Keep these exact mappings:
#     "B Education in Primary Schools (age 4 - 12) (day)"     -> "Primary_Education_Day"
#     "B Education in Primary Schools (age 4 - 12) (ALPO)"    -> "Primary_Education_ALPO"
#     "B Education in Primary Schools (age 4 - 12) (Regular)" -> "Primary_Education_Regular"
#     "B Arts Therapies (Music Therapy)"                      -> "Music_Therapy"
#     "B Arts Therapies (Drama Therapy)"                      -> "Drama_Therapy"
#     "B Arts Therapies (Art Therapy)"                        -> "Art_Therapy"
# - Remove leading degree prefixes: "B " and "Bachelor of "
# - Correct known typos (e.g., "Chemics" -> "Chemistry")
# - Remove prepositions between program names: "and", "with", "in" (case-insensitive)
# - Remove commas and ampersands
# - Normalize whitespace and apply Title Case
# - Convert spaces between words into underscores at the end
# - Preserve "Teacher" as-is
# - Map NaN to "Unknown"

col = "sdo5_degree"

def clean_degree(value):
    if pd.isna(value):
        return "Unknown"
    
    v = str(value).strip()

    # ---- Exact mappings to keep as specified ----
    exact_map = {
        "B Education in Primary Schools (age 4 - 12) (day)": "Primary_Education_Day",
        "B Education in Primary Schools (age 4 - 12) (ALPO)": "Primary_Education_ALPO",
        "B Education in Primary Schools (age 4 - 12) (Regular)": "Primary_Education_Regular",
        "B Arts Therapies (Music Therapy)": "Music_Therapy",
        "B Arts Therapies (Drama Therapy)": "Drama_Therapy",
        "B Arts Therapies (Art Therapy)": "Art_Therapy",
    }
    if v in exact_map:
        return exact_map[v]

    # ---- Remove prefixes ----
    v = re.sub(r"^\s*B\s+", "", v)
    v = re.sub(r"^\s*Bachelor of\s+", "", v, flags=re.IGNORECASE)

    # ---- Fix known inconsistencies ----
    v = v.replace("Chemics", "Chemistry")

    # ---- Remove prepositions and punctuation ----
    v = v.replace("&", " ")
    v = re.sub(r"\b(?:and|with|in)\b", "", v, flags=re.IGNORECASE)
    v = v.replace(",", " ")

    # ---- Normalize whitespace and casing ----
    v = re.sub(r"\s+", " ", v).strip()
    v = v.title()

    # ---- Replace spaces with underscores ----
    v = v.replace(" ", "_")

    return v

# Apply cleaning directly to the original column
data[col] = data[col].apply(clean_degree)

# Optional check
print(data[col].value_counts())


In [ ]:
# Save version 2 data here to later combine it with version 1
data.to_csv(os.path.join("..", "data", "processed", "dropout_version2_data.csv"), header=True)